In [1]:
# [1] 필요한 라이브러리 불러오기 및 .env 파일에서 API Key 로딩

from dotenv import load_dotenv
import os

# .env 파일에서 환경변수 불러오기
load_dotenv()

# KAMIS-like API key 변수명
API_KEY = os.getenv("KAMIS_API_KEY")

# 확인용 출력 (주의: 실제 배포 시에는 노출 금지)
print("API Key Loaded:", "✅" if API_KEY else "❌ 오류: API Key 없음")



API Key Loaded: ✅


In [6]:
# [11] 전체 품목 중 '배추', '쌀', '양파', '상추', '사과' 포함된 중분류 이름/코드 자동 추출

import requests
import xml.etree.ElementTree as ET

def auto_find_middle_codes(api_key, keywords, chunk_size=1000, max_rows=13249):
    base_url = "http://211.237.50.150:7080/openapi"
    grid_id = "Grid_20141221000000000120_1"
    
    found = {}
    
    for start in range(1, max_rows + 1, chunk_size):
        end = min(start + chunk_size - 1, max_rows)
        url = f"{base_url}/{api_key}/xml/{grid_id}/{start}/{end}"

        try:
            response = requests.get(url)
            response.raise_for_status()
            root = ET.fromstring(response.text)
            rows = root.findall(".//row")
            
            for row in rows:
                name = row.findtext("PRDLST_NM")
                code = row.findtext("PRDLST_CD")
                for keyword in keywords:
                    if keyword in name and keyword not in found:
                        found[keyword] = {"품목명": name, "코드": code}
                        print(f"✅ {keyword} → {name} (코드: {code})")
                if len(found) == len(keywords):
                    break
        except Exception as e:
            print(f"❌ 오류: {e}")
            break
        
        if len(found) == len(keywords):
            break

    return found

# ✅ 실행
keywords = ["쌀", "배추", "양파", "상추", "사과"]
code_dict = auto_find_middle_codes(API_KEY, keywords)

print("\n📌 최종 중분류 코드 매핑 결과:")
for k, v in code_dict.items():
    print(f"{k}: {v['품목명']} (코드: {v['코드']})")


✅ 사과 → 사과 (코드: 19I9)
✅ 양파 → 양파 (코드: 1201)
✅ 배추 → 양배추 (코드: 1004)
✅ 상추 → 상추 (코드: 1005)
✅ 쌀 → 쌀 (코드: 0103)

📌 최종 중분류 코드 매핑 결과:
사과: 사과 (코드: 19I9)
양파: 양파 (코드: 1201)
배추: 양배추 (코드: 1004)
상추: 상추 (코드: 1005)
쌀: 쌀 (코드: 0103)
